In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
# 처음이면 clone, 이미 있으면 pull
!git clone https://github.com/chaeyoungwon/mask-aware-inpainting.git
# 또는
# !git checkout yeonholee
%cd mask-aware-inpainting


## 데이터셋 구성

In [ ]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

# 설치 및 인증
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jessicali9530/celeba-dataset -p ./mask-aware-inpainting/data/
!cd mask-aware-inpainting/data && unzip celeba-dataset.zip -d celeba_temp

In [ ]:
import os, shutil

base = './mask-aware-inpainting/data'
os.makedirs(f'{base}/celeba/img_align_celeba', exist_ok=True)

# 이미지 이동
!mv {base}/celeba_temp/img_align_celeba/* {base}/celeba/img_align_celeba/

# 메타 파일 이동 (list_eval_partition.txt 등)
!mv {base}/celeba_temp/*.txt {base}/celeba/
!mv {base}/celeba_temp/*.csv {base}/celeba/ 2>/dev/null || true

In [ ]:
!ls ./mask-aware-inpainting/data/celeba/

In [ ]:
!python3 prepare_fixed_testset.py

In [ ]:
# conv_type을 'pconv'으로 설정
import re
with open('config.py', 'r') as f:
    _cfg = f.read()
_cfg = re.sub(r"conv_type\s*=\s*'[^']*'", "conv_type = 'pconv'", _cfg)
with open('config.py', 'w') as f:
    f.write(_cfg)
print("conv_type set to 'pconv'")


## 학습

In [ ]:
!python3 train.py

In [ ]:
!python evaluate.py checkpoints/pconv/best_model.pth --conv_type pconv --fixed_testset ./fixed_testset


## 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('checkpoints/pconv/training_history_pconv.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('PConv CAE', fontsize=14)

ax1.plot(df['epoch'], df['val_psnr'], color='steelblue')
ax1.set_title('Epoch별 Val PSNR')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('PSNR (dB)'); ax1.grid(True)

ax2.plot(df['epoch'], df['val_ssim'], color='goldenrod')
ax2.set_title('Epoch별 Val SSIM')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('SSIM'); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f'최종 PSNR: {df["val_psnr"].iloc[-1]:.2f} dB  |  최종 SSIM: {df["val_ssim"].iloc[-1]:.4f}')
